In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
PROJECT_PATH = "/content/drive/MyDrive/Smart-Retail-Sales-Customer-Intelligence"

import os
os.chdir(PROJECT_PATH)

print(os.getcwd())
file_path = "data/raw/online_retail_II.xlsx"
import os

print(os.path.exists(file_path))

/content/drive/MyDrive/Smart-Retail-Sales-Customer-Intelligence
True


In [3]:
import pandas as pd
df=pd.read_excel(file_path)
df_2009_2010=pd.read_excel(file_path , sheet_name = 0)
df_2010_2011 = pd.read_excel(file_path , sheet_name =1)

In [ ]:
df_2010_2011.isnull().sum()
df_2009_2010.isnull().sum()

,0
Invoice,0
StockCode,0
Description,2928
Quantity,0
InvoiceDate,0
Price,0
Customer ID,107927
Country,0


In [ ]:
df_2009_2010[df_2009_2010['Description'].isnull()]
df_2010_2011[df_2010_2011['Description'].isnull()]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1510,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1985,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1986,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
2022,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535322,581199,84581,NaN,-2,2011-12-07 18:26:00,0.0,NaN,United Kingdom
535326,581203,23406,NaN,15,2011-12-07 18:31:00,0.0,NaN,United Kingdom
535332,581209,21620,NaN,6,2011-12-07 18:35:00,0.0,NaN,United Kingdom
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom


In [ ]:
df_2009_2010[df_2009_2010['StockCode']==22139][["StockCode", "Description"]]
df_2010_2011[df_2010_2011['StockCode']==22139][["StockCode","Description"]]

,StockCode,Description
73,22139,RETRO SPOT TEA SET CERAMIC 11 PC
220,22139,RETRO SPOT TEA SET CERAMIC 11 PC
324,22139,RETRO SPOT TEA SET CERAMIC 11 PC
778,22139,RETRO SPOT TEA SET CERAMIC 11 PC
942,22139,RETRO SPOT TEA SET CERAMIC 11 PC
...,...,...
517962,22139,RETROSPOT TEA SET CERAMIC 11 PC
520306,22139,RETROSPOT TEA SET CERAMIC 11 PC
522828,22139,RETROSPOT TEA SET CERAMIC 11 PC
523269,22139,NaN


In [ ]:
missing_desc=df_2009_2010.loc[df_2009_2010['Description'].isnull(),"StockCode"]
missing_desc.head(20)

,StockCode
470,21646
3114,20683
3161,21350
3731,84292
4296,18010
4566,85049G
6378,35751C
6555,79323G
6576,21098
6581,21166


In [ ]:
missing_stockcodes=df_2009_2010.loc[df_2009_2010['Description'].isnull(),'StockCode'].unique()
len(missing_stockcodes)
valid_stockcode=df_2009_2010.loc[df_2009_2010['Description'].notnull(),'StockCode'].unique()
len(valid_stockcode)
common_stockcode=set(missing_stockcodes) & set(valid_stockcode)
len(common_stockcode)

1564

In [ ]:
missing_stockcodes=df_2010_2011.loc[df_2010_2011['Description'].isnull(),'StockCode'].unique()
len(missing_stockcodes)
valid_stockcode=df_2010_2011.loc[df_2010_2011['Description'].notnull(),'StockCode'].unique()
len(valid_stockcode)
common_stockcode=set(missing_stockcodes) & set(valid_stockcode)
len(common_stockcode)

848

In [ ]:
df_2009_2010.loc[
    (df_2009_2010["StockCode"] == 22139) &
    (df_2009_2010["Description"].notnull()),
    "Description"
].unique()
df_2009_2010.loc[df_2009_2010['StockCode']==22139, ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price"]]
df_2009_2010.loc[
    (df_2009_2010["StockCode"] == 22139) &
    (df_2009_2010["Description"].notnull()),
    "Description"
].value_counts()
df_2009_2010.loc[
    (df_2009_2010["StockCode"] == 22139) &
    (df_2009_2010["Description"].isnull()),
    "Description"
].shape[0]
mask = (
    (df_2009_2010["StockCode"] == 22139) &
    (df_2009_2010["Description"].isnull())
)
df_2009_2010.loc[mask, "Description"]
df_2009_2010.loc[mask, "Description"] = "RETRO SPOT TEA SET CERAMIC 11 PC"
df_2009_2010.loc[
    (df_2009_2010["StockCode"] == 22139) &
    (df_2009_2010["Description"].isnull())
].shape[0]


In [ ]:
df_2009_2010.groupby('StockCode')['Description'].value_counts()
desc_counts = (
    df_2009_2010
    .groupby("StockCode")["Description"]
    .value_counts()
)
most_common = desc_counts.groupby(level=0).idxmax()
most_common.head(10)
most_common = most_common.str[1]
most_common.head(10)
top_count = desc_counts.groupby(level=0).max()

top_count.head(10)
total_count = desc_counts.groupby(level=0).sum()

total_count.head(10)
dominance_ratio = top_count / total_count

dominance_ratio.head(10)
print("100%:", (dominance_ratio == 1.0).sum())
print("90%+:", (dominance_ratio >= 0.90).sum())
print("80%+:", (dominance_ratio >= 0.80).sum())
safe_stockcodes = dominance_ratio[dominance_ratio >= 0.90].index
len(safe_stockcodes)
safe_descriptions = most_common.loc[safe_stockcodes]
safe_descriptions.head(10)
missing_mask = (
    df_2009_2010["Description"].isnull()
    & df_2009_2010["StockCode"].isin(safe_stockcodes)
)

missing_mask.sum()
df_2009_2010.loc[missing_mask, "Description"] = (
    df_2009_2010.loc[missing_mask, "StockCode"].map(safe_descriptions)
)
df_2009_2010["Description"].isnull().sum()
remaining_missing = df_2009_2010[
    df_2009_2010["Description"].isnull()
]

remaining_missing["StockCode"].nunique()

100%: 3589
90%+: 3913
80%+: 4008


433

In [ ]:
remaining_stockcodes = remaining_missing["StockCode"].unique()

remaining_valid_stockcodes = df_2009_2010.loc[
    df_2009_2010["Description"].notnull(),
    "StockCode"
].unique()

no_description_stockcodes = set(remaining_stockcodes) - set(remaining_valid_stockcodes)

len(no_description_stockcodes)
low_confidence_stockcodes = (
    set(remaining_stockcodes)
    & set(remaining_valid_stockcodes)
)

len(low_confidence_stockcodes)
low_confidence_stockcodes = list(low_confidence_stockcodes)

low_confidence_ratio = dominance_ratio.loc[
    low_confidence_stockcodes
].sort_values()

low_confidence_ratio.head(20)
df_2009_2010["Description"].isnull().sum()
df_2009_2010.duplicated().sum()
duplicates = df_2009_2010[
    df_2009_2010.duplicated(keep=False)
]

duplicates.head(20)
duplicates.shape
df_2009_2010.duplicated(keep=False).sum()
duplicates.head(10)
duplicates[
    duplicates.duplicated(keep=False)
].head(20)
df_2009_2010 = df_2009_2010.drop_duplicates()
df_2009_2010.duplicated().sum()
df_2009_2010.shape

(518596, 8)

In [ ]:
df_2009_2010.loc[df_2009_2010["Description"].isnull(), "StockCode"].head(20)
df_2009_2010[df_2009_2010["StockCode"].isin(
    df_2009_2010.loc[df_2009_2010["Description"].isnull(), "StockCode"]
)][["StockCode", "Description"]].head(30)
df_2009_2010.groupby("StockCode")["Description"].nunique().sort_values(ascending=False).head(20)
df_2009_2010[df_2009_2010["StockCode"] == 22139]["Description"].value_counts()
df_2009_2010.loc[
    df_2009_2010["StockCode"] == 22139,
    "Description"
] = "RETRO SPOT TEA SET CERAMIC 11 PC"
df_2009_2010[df_2009_2010["StockCode"] == 22139]["Description"].value_counts()

,count
Description,
RETRO SPOT TEA SET CERAMIC 11 PC,1203


In [ ]:
df_2009_2010['Description'].isnull().sum()
missing_stockcodes = df_2009_2010.loc[
    df_2009_2010["Description"].isnull(),
    "StockCode"
]

missing_stockcodes.nunique()
available_descriptions = df_2009_2010[
    df_2009_2010["Description"].notnull()
].groupby("StockCode")["Description"].nunique()

available_descriptions.loc[
    available_descriptions
    .index.isin(missing_stockcodes)
].value_counts()
single_description_codes = available_descriptions[
    available_descriptions == 1
].index

description_mapping = (
    df_2009_2010[df_2009_2010["Description"].notnull()]
    .groupby("StockCode")["Description"]
    .first()
)

df_2009_2010.loc[
    df_2009_2010["Description"].isnull() &
    df_2009_2010["StockCode"].isin(single_description_codes),
    "Description"
] = df_2009_2010.loc[
    df_2009_2010["Description"].isnull() &
    df_2009_2010["StockCode"].isin(single_description_codes),
    "StockCode"
].map(description_mapping)
df_2009_2010["Description"].isnull().sum()

remaining_missing_codes = df_2009_2010.loc[
    df_2009_2010["Description"].isnull(),
    "StockCode"
]

remaining_missing_codes.nunique()
remaining_codes_info = (
    df_2009_2010.groupby("StockCode")["Description"]
    .nunique(dropna=True)
)

remaining_codes_info.loc[
    remaining_codes_info.index.isin(remaining_missing_codes)
].value_counts()

no_description_codes = remaining_codes_info.loc[
    remaining_codes_info.index.isin(remaining_missing_codes) &
    (remaining_codes_info == 0)
].index

len(no_description_codes)

df_2009_2010[
    df_2009_2010["StockCode"].isin(no_description_codes)
][["StockCode", "Quantity", "Price", "Customer ID", "Country"]].head(20)

df_2009_2010[
    df_2009_2010["StockCode"].isin(no_description_codes)
].shape
df_2009_2010.loc[
    df_2009_2010["Description"].isnull() &
    df_2009_2010["StockCode"].isin(no_description_codes),
    "Description"
] = "Unknown"
df_2009_2010["Description"].isnull().sum()

np.int64(405)

In [ ]:
multi_description_codes = remaining_codes_info.loc[
    remaining_codes_info.index.isin(remaining_missing_codes) &
    (remaining_codes_info > 1)
].index

df_2009_2010[
    df_2009_2010["StockCode"].isin(multi_description_codes)
].groupby(["StockCode", "Description"]).size().sort_values(ascending=False).head(30)
df_2009_2010[
    df_2009_2010["StockCode"] == 22423
][["StockCode", "Description"]].value_counts()
description_counts = (
    df_2009_2010[
        df_2009_2010["Description"].notnull()
    ]
    .groupby(["StockCode", "Description"])
    .size()
    .reset_index(name="Count")
    .sort_values(["StockCode", "Count"], ascending=[True, False])
)
missing_description_rows = df_2009_2010[
    df_2009_2010["Description"].isnull()
]

missing_description_rows[
    ["StockCode", "Quantity", "Price", "Customer ID", "Country"]
].head(20)
df_2009_2010[
    df_2009_2010["Description"].isnull()
]["Customer ID"].isnull().sum()
df_2009_2010[
    df_2009_2010["Description"].isnull()
]["Customer ID"].value_counts(dropna=False).head()
df_2009_2010[
    df_2009_2010["Description"].isnull()
]["Customer ID"].value_counts(dropna=False).head()
df_2009_2010[
    df_2009_2010["Description"].isnull()
]["StockCode"].nunique()
description_summary = (
    df_2009_2010[
        df_2009_2010["Description"].notnull()
    ]
    .groupby(["StockCode", "Description"])
    .size()
    .reset_index(name="Count")
)

description_summary[
    description_summary["StockCode"].isin(remaining_missing_codes)
].sort_values(
    ["StockCode", "Count"],
    ascending=[True, False]
).head(30)
description_summary[
    description_summary["StockCode"].isin(remaining_missing_codes)
].groupby("StockCode")["Count"].max().describe()
dominant_counts = (
    description_summary[
        description_summary["StockCode"].isin(remaining_missing_codes)
    ]
    .groupby("StockCode")["Count"]
    .max()
)

dominant_counts.value_counts().sort_index()
dominant_ratio = (
    description_summary[
        description_summary["StockCode"].isin(remaining_missing_codes)
    ]
    .groupby("StockCode")["Count"]
    .agg(["max", "sum"])
)

dominant_ratio["ratio"] = (
    dominant_ratio["max"] / dominant_ratio["sum"]
)

dominant_ratio.sort_values("ratio", ascending=False).head(20)
(dominant_ratio["ratio"] >= 0.95).sum()
(dominant_ratio["ratio"] < 0.95).sum()
reliable_codes = dominant_ratio[
    dominant_ratio["ratio"] >= 0.95
].index
dominant_descriptions = (
    description_summary[
        description_summary["StockCode"].isin(reliable_codes)
    ]
    .sort_values(["StockCode", "Count"], ascending=[True, False])
    .drop_duplicates("StockCode")
    .set_index("StockCode")["Description"]
)
mask = (
    df_2009_2010["Description"].isnull() &
    df_2009_2010["StockCode"].isin(reliable_codes)
)

df_2009_2010.loc[mask, "Description"] = (
    df_2009_2010.loc[mask, "StockCode"].map(dominant_descriptions)
)
df_2009_2010["Description"].isnull().sum()
df_2009_2010[
    df_2009_2010["Description"].isnull()
][["StockCode", "Quantity", "Price", "Customer ID", "Country"]]
df_2009_2010.loc[
    df_2009_2010["Description"].isnull(),
    "Description"
] = "Unknown"
df_2009_2010["Description"].isnull().sum()

In [ ]:
df_2009_2010.isnull().sum()
df_2009_2010.duplicated().sum()
df_2009_2010[df_2009_2010.duplicated(keep=False)].head(20)
df_2009_2010 = df_2009_2010.drop_duplicates()
df_2009_2010.duplicated().sum()
df_2009_2010.shape

(518596, 8)

In [ ]:
df_2009_2010.dtypes

,0
Invoice,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[ns]
Price,float64
Customer ID,float64
Country,object


In [ ]:
df_2009_2010["Quantity"].describe()
df_2009_2010[df_2009_2010["Quantity"] < 0].shape
df_2009_2010[
    df_2009_2010["Quantity"] < 0
][["Invoice", "StockCode", "Description", "Quantity", "Price"]].head(20)
df_2009_2010[
    df_2009_2010["Invoice"].astype(str).str.startswith("C")
].shape
df_2009_2010[
    (df_2009_2010["Quantity"] < 0) &
    (~df_2009_2010["Invoice"].astype(str).str.startswith("C"))
][["Invoice", "StockCode", "Description", "Quantity", "Price"]].head(20)
negative_qty = df_2009_2010["Quantity"] < 0

cancellation = (
    negative_qty &
    df_2009_2010["Invoice"].astype(str).str.startswith("C")
)

other_negative = (
    negative_qty &
    ~df_2009_2010["Invoice"].astype(str).str.startswith("C")
)

print("Negative Quantity:", negative_qty.sum())
print("Cancellation:", cancellation.sum())
print("Other Negative Transactions:", other_negative.sum())
df_2009_2010["Price"].describe()
df_2009_2010[df_2009_2010["Price"] < 0].shape
df_2009_2010[
    df_2009_2010["Price"] < 0
][["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "Country"]]
df_2009_2010 = df_2009_2010[
    df_2009_2010["Price"] >= 0
]
df_2009_2010[df_2009_2010["Price"] < 0].shape
df_2009_2010[df_2009_2010["Price"] == 0].shape
df_2009_2010[
    df_2009_2010["Price"] == 0
][["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "Country"]].head(20)
df_2009_2010[df_2009_2010["Price"] == 0]["Quantity"].gt(0).sum()
df_2009_2010[df_2009_2010["Price"] == 0]["Quantity"].lt(0).sum()
df_2009_2010[df_2009_2010["Price"] > 1000][
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]
].head(20)
df_2009_2010["SalesAmount"] = (
    df_2009_2010["Quantity"] * df_2009_2010["Price"]
)
df_2009_2010["SalesAmount"] = (
    df_2009_2010["Quantity"] * df_2009_2010["Price"]
)
df_2009_2010[
    ["Quantity", "Price", "SalesAmount"]
].head(10)

Negative Quantity: 12302
Cancellation: 10181
Other Negative Transactions: 2121


,Quantity,Price,SalesAmount
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0
5,24,1.65,39.6
6,24,1.25,30.0
7,10,5.95,59.5
8,12,2.55,30.6
9,12,3.75,45.0


In [ ]:
os.makedirs("data/processed", exist_ok=True)
df_2009_2010["Invoice"] = df_2009_2010["Invoice"].astype(str)
df_2009_2010["StockCode"] = df_2009_2010["StockCode"].astype(str)
df_2009_2010["Description"] = df_2009_2010["Description"].astype(str)
df_2009_2010.to_parquet(
    "data/processed/online_retail_cleaned.parquet",
    index=False
)
os.path.exists("data/processed/online_retail_cleaned.parquet")

True

In [ ]:
df_2010_2011.shape
df_2010_2011.info()
df_2010_2011.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  object        
 1   StockCode    541910 non-null  object        
 2   Description  540456 non-null  object        
 3   Quantity     541910 non-null  int64         
 4   InvoiceDate  541910 non-null  datetime64[ns]
 5   Price        541910 non-null  float64       
 6   Customer ID  406830 non-null  float64       
 7   Country      541910 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


,0
Invoice,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
Price,0
Customer ID,135080
Country,0


In [ ]:
df_2010_2011[df_2010_2011["Description"].isna()]["StockCode"].nunique()
df_2010_2011[df_2010_2011["Description"].isna()]["StockCode"].value_counts().head(20)
missing_codes = df_2010_2011.loc[
    df_2010_2011["Description"].isna(), "StockCode"
].unique()

description_check = (
    df_2010_2011[df_2010_2011["StockCode"].isin(missing_codes)]
    .groupby("StockCode")["Description"]
    .nunique()
    .sort_values()
)

description_check.value_counts().sort_index()
df_2010_2011[
    df_2010_2011["StockCode"].isin(
        description_check[description_check == 1].index
    )
][["StockCode", "Description"]].dropna().drop_duplicates().head(20)
description_map = (
    df_2010_2011.dropna(subset=["Description"])
    .groupby("StockCode")["Description"]
    .first()
)

mask = (
    df_2010_2011["Description"].isna()
    & df_2010_2011["StockCode"].isin(
        description_check[description_check == 1].index
    )
)

df_2010_2011.loc[mask, "Description"] = (
    df_2010_2011.loc[mask, "StockCode"].map(description_map)
)
df_2010_2011["Description"].isna().sum()


np.int64(421)

In [ ]:
remaining_missing = df_2010_2011[
    df_2010_2011["Description"].isna()
]

remaining_missing["StockCode"].nunique()
remaining_missing["StockCode"].value_counts().head(20)
remaining_description_check = (
    df_2010_2011[
        df_2010_2011["StockCode"].isin(
            remaining_missing["StockCode"]
        )
    ]
    .groupby("StockCode")["Description"]
    .nunique()
)

remaining_description_check.value_counts().sort_index()
remaining_description_check.size
remaining_description_check.isna().sum()
remaining_description_check[
    ~remaining_description_check.isin([0, 2, 3, 4])
]
no_description_codes = remaining_description_check[
    remaining_description_check == 0
].index

len(no_description_codes)
mask_unknown = (
    df_2010_2011["Description"].isna()
    & df_2010_2011["StockCode"].isin(no_description_codes)
)

df_2010_2011.loc[mask_unknown, "Description"] = "Unknown"
df_2010_2011["Description"].isna().sum()

np.int64(309)

In [ ]:
multi_desc_codes = remaining_description_check[
    remaining_description_check > 1
].index

desc_counts = (
    df_2010_2011[
        df_2010_2011["StockCode"].isin(multi_desc_codes)
        & df_2010_2011["Description"].notna()
    ]
    .groupby(["StockCode", "Description"])
    .size()
)

dominant_ratio = (
    desc_counts.groupby(level=0).max()
    / desc_counts.groupby(level=0).sum()
)

dominant_ratio.sort_values().head(20)
dominant_ratio.describe()
(dominant_ratio >= 0.95).sum(), (dominant_ratio < 0.95).sum()
reliable_codes = dominant_ratio[dominant_ratio >= 0.95].index

dominant_description = (
    df_2010_2011[
        df_2010_2011["StockCode"].isin(reliable_codes)
        & df_2010_2011["Description"].notna()
    ]
    .groupby("StockCode")["Description"]
    .agg(lambda x: x.value_counts().index[0])
)

len(dominant_description)
mask_reliable = (
    df_2010_2011["Description"].isna()
    & df_2010_2011["StockCode"].isin(reliable_codes)
)

df_2010_2011.loc[mask_reliable, "Description"] = (
    df_2010_2011.loc[mask_reliable, "StockCode"]
    .map(dominant_description)
)
df_2010_2011["Description"].isna().sum()
remaining_missing = df_2010_2011[
    df_2010_2011["Description"].isna()
]

remaining_missing["StockCode"].value_counts()
remaining_missing["StockCode"].value_counts().to_string()
ambiguous_details = (
    df_2010_2011[
        df_2010_2011["StockCode"].isin(remaining_missing["StockCode"])
        & df_2010_2011["Description"].notna()
    ]
    .groupby(["StockCode", "Description"])
    .size()
    .reset_index(name="Count")
    .sort_values(["StockCode", "Count"], ascending=[True, False])
)

ambiguous_details
print(ambiguous_details.to_string(index=False))
ignore_descriptions = [
    "damaged",
    "damages",
    "check",
    "?",
    "found",
    "Adjustment",
    "adjustment",
    "label mix up",
    "Amazon",
    "Found in w/hse",
    "alan hodge cant mamage this section",
    "did  a credit  and did not tick ret"
]

clean_desc_counts = (
    ambiguous_details[
        ~ambiguous_details["Description"].isin(ignore_descriptions)
    ]
)

dominant_product_description = (
    clean_desc_counts
    .sort_values(["StockCode", "Count"], ascending=[True, False])
    .groupby("StockCode")
    .first()["Description"]
)

dominant_product_description
mask_remaining = df_2010_2011["Description"].isna()

df_2010_2011.loc[mask_remaining, "Description"] = (
    df_2010_2011.loc[mask_remaining, "StockCode"]
    .map(dominant_product_description)
)
df_2010_2011["Description"].isna().sum()
df_2010_2011["Customer ID"].isna().sum()
df_2010_2011.duplicated().sum()
df_2010_2011 = df_2010_2011.drop_duplicates()
df_2010_2011.duplicated().sum()
df_2010_2011.shape
df_2010_2011.dtypes

Empty DataFrame
Columns: [StockCode, Description, Count]
Index: []


,0
Invoice,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[ns]
Price,float64
Customer ID,float64
Country,object


In [ ]:
df_2010_2011["Quantity"].describe()
negative_quantity = (
    df_2010_2011["Quantity"] < 0
).sum()

negative_quantity
cancellation_count = (
    df_2010_2011["Invoice"]
    .astype(str)
    .str.startswith("C")
    .sum()
)

negative_quantity, cancellation_count
other_negative = df_2010_2011[
    (df_2010_2011["Quantity"] < 0)
    & (~df_2010_2011["Invoice"].astype(str).str.startswith("C"))
]

other_negative["Description"].value_counts().head(20)
df_2010_2011["Price"].describe()
(df_2010_2011["Price"] < 0).sum()
df_2010_2011[df_2010_2011["Price"] < 0]
df_2010_2011 = df_2010_2011[df_2010_2011["Price"] >= 0]
(df_2010_2011["Price"] < 0).sum()

np.int64(0)

In [ ]:
df_2010_2011["SalesAmount"] = (
    df_2010_2011["Quantity"] * df_2010_2011["Price"]
)
df_2010_2011[["Quantity", "Price", "SalesAmount"]].head()
df_2010_2011["SalesAmount"].describe()
df_2010_2011.shape
df_2010_2011.isna().sum()

,0
Invoice,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
Price,0
Customer ID,135035
Country,0
SalesAmount,0


In [ ]:
for col in df_2010_2011.select_dtypes(include="object").columns:
    df_2010_2011[col] = df_2010_2011[col].astype(str)
    output_path = (
    "/content/drive/MyDrive/"
    "Smart-Retail-Sales-Customer-Intelligence/"
    "data/processed/retail_2010_2011_cleaned.parquet"
)

df_2010_2011.to_parquet(output_path, index=False)
df_retail = pd.concat(
    [df_2009_2010, df_2010_2011],
    ignore_index=True
)
df_retail.shape
df_retail.columns
df_retail.isna().sum()
df_retail.duplicated().sum()
df_2009_2010.shape
df_2009_2010.columns
df_2009_2010["SalesAmount"] = (
    df_2009_2010["Quantity"] * df_2009_2010["Price"]
)
df_2009_2010 = df_2009_2010.drop_duplicates()
df_2009_2010.shape
df_retail = pd.concat(
    [df_2009_2010, df_2010_2011],
    ignore_index=True
)
df_retail.shape
df_retail.duplicated().sum()
df_retail.isna().sum()


NameError: name 'clean_2009_2010' is not defined

In [5]:
import os

processed_path = (
    "/content/drive/MyDrive/"
    "Smart-Retail-Sales-Customer-Intelligence/"
    "data/processed"
)

os.listdir(processed_path)
processed_path = (
    "/content/drive/MyDrive/"
    "Smart-Retail-Sales-Customer-Intelligence/"
    "data/processed"
)

df_2009_clean = pd.read_parquet(
    f"{processed_path}/online_retail_cleaned.parquet"
)

df_2010_clean = pd.read_parquet(
    f"{processed_path}/retail_2010_2011_cleaned.parquet"
)
df_2009_clean.shape, df_2010_clean.shape
df_retail = pd.concat(
    [df_2009_clean, df_2010_clean],
    ignore_index=True
)
df_retail.shape
df_retail.duplicated().sum()
df_retail.isna().sum()
df_retail.duplicated().sum()

np.int64(22174)

In [13]:
duplicate_rows = df_retail[df_retail.duplicated(keep=False)]

duplicate_rows.head(20)
duplicate_rows.shape
df_retail = df_retail.drop_duplicates().reset_index(drop=True)
df_retail.duplicated().sum()
df_retail.shape
df_retail.isnull().sum()
df_retail.dtypes

,0
Invoice,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[ns]
Price,float64
Customer ID,float64
Country,object
SalesAmount,float64


In [14]:
final_path = (
    "/content/drive/MyDrive/"
    "Smart-Retail-Sales-Customer-Intelligence/"
    "data/processed/retail_sales_cleaned.parquet"
)

df_retail.to_parquet(final_path, index=False)
import os

os.path.exists(final_path)

True